가중치개수와 연산횟수는?

In [4]:
###########################################################
#Naive 버전
############################################################
import torch
import torch.nn as nn

class CustomModel(nn.Module):
    def __init__(self):
        super(CustomModel, self).__init__()
        
        # [해설] Keras의 Dense(192)가 3차원 텐서(H, W, C)에 적용될 경우, 
        # 각 위치(pixel)별 채널(C=3)에만 선형 변환이 적용되므로 
        # PyTorch에서는 1x1 합성곱(Conv2d)과 정확히 동일한 역할을 합니다.
        self.pre_conv = nn.Conv2d(in_channels=3, out_channels=192, kernel_size=1)
        
        # 1번 브랜치: 1x1 Conv (padding='same' -> padding=0)
        self.conv1 = nn.Conv2d(in_channels=192, out_channels=64, kernel_size=1, padding=0)
        
        # 2번 브랜치: 3x3 Conv (padding='same' -> kernel_size=3 이므로 padding=1)
        self.conv2 = nn.Conv2d(in_channels=192, out_channels=128, kernel_size=3, padding=1)
        
        # 3번 브랜치: 5x5 Conv (padding='same' -> kernel_size=5 이므로 padding=2)
        self.conv3 = nn.Conv2d(in_channels=192, out_channels=32, kernel_size=5, padding=2)
        
        # 4번 브랜치: MaxPool (pool_size=3, strides=1, padding='same' -> padding=1)
        self.pool = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        # 입력 텐서 전처리 (Dense 레이어 역할 대체)
        pre = self.pre_conv(x)
        
        # 병렬 브랜치 연산 수행
        c1 = self.conv1(pre)
        c2 = self.conv2(pre)
        c3 = self.conv3(pre)
        p = self.pool(pre)
        
        # 채널 방향으로 결합 
        # (Keras의 axis=-1 은 PyTorch에서 채널 차원인 dim=1에 해당합니다)
        out = torch.cat([c1, c2, c3, p], dim=1)
        return out

# 1. 모델 인스턴스 생성
model = CustomModel()
print(model)

# 2. 더미 입력 데이터로 순전파 테스트 
# Keras: (batch, 28, 28, 3) -> PyTorch: (batch, Channels, Height, Width) = (1, 3, 28, 28)
dummy_input = torch.randn(1, 3, 28, 28)
output = model(dummy_input)

print(f"\n입력 텐서 크기: {dummy_input.shape}")
print(f"출력 텐서 크기: {output.shape}")  # 예상 출력: (1, 416, 28, 28) -> (64 + 128 + 32 + 192 = 416채널)

CustomModel(
  (pre_conv): Conv2d(3, 192, kernel_size=(1, 1), stride=(1, 1))
  (conv1): Conv2d(192, 64, kernel_size=(1, 1), stride=(1, 1))
  (conv2): Conv2d(192, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(192, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (pool): MaxPool2d(kernel_size=3, stride=1, padding=1, dilation=1, ceil_mode=False)
)

입력 텐서 크기: torch.Size([1, 3, 28, 28])
출력 텐서 크기: torch.Size([1, 416, 28, 28])


1x1 Conv층을추가한인셉션모듈구현

In [3]:
import torch
import torch.nn as nn

class InceptionBlock(nn.Module):
    def __init__(self):
        super(InceptionBlock, self).__init__()
        
        # Keras의 Dense(192)는 3차원 텐서(H, W, C)에서 채널 방향의 선형 변환이므로 PyTorch에서는 1x1 Conv로 대체됩니다.
        self.pre_conv = nn.Conv2d(in_channels=3, out_channels=192, kernel_size=1)
        
        # 1번 브랜치: 1x1 Conv
        self.conv1 = nn.Conv2d(in_channels=192, out_channels=64, kernel_size=1, padding=0)
        
        # 2번 브랜치: 1x1 Conv -> 3x3 Conv
        self.conv1_2 = nn.Conv2d(in_channels=192, out_channels=96, kernel_size=1, padding=0)
        self.conv2 = nn.Conv2d(in_channels=96, out_channels=128, kernel_size=3, padding=1)
        
        # 3번 브랜치: 1x1 Conv -> 5x5 Conv
        self.conv1_3 = nn.Conv2d(in_channels=192, out_channels=16, kernel_size=1, padding=0)
        self.conv3 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=5, padding=2)
        
        # 4번 브랜치: MaxPool -> 1x1 Conv
        self.pool = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
        self.conv1_4 = nn.Conv2d(in_channels=192, out_channels=32, kernel_size=1, padding=0)

    def forward(self, x):
        # 입력 전처리 (Dense 192 역할)
        pre = self.pre_conv(x)
        
        # 1번 브랜치 연산
        c1 = self.conv1(pre)
        
        # 2번 브랜치 연산
        c1_2 = self.conv1_2(pre)
        c2 = self.conv2(c1_2)
        
        # 3번 브랜치 연산
        c1_3 = self.conv1_3(pre)
        c3 = self.conv3(c1_3)
        
        # 4번 브랜치 연산
        p = self.pool(pre)
        c1_4 = self.conv1_4(p)
        
        # 모든 브랜치 출력 결과들을 채널 차원(dim=1)을 기준으로 결합 (Concatenate)
        # 총 채널 수: 64 + 128 + 32 + 32 = 256
        out = torch.cat([c1, c2, c3, c1_4], dim=1)
        return out

# 1. 모델 인스턴스 생성
model = InceptionBlock()
print(model)

# 2. 더미 입력 데이터로 순전파 테스트
# PyTorch 형식: (Batch=1, Channels=3, Height=28, Width=28)
dummy_input = torch.randn(1, 3, 28, 28)
output = model(dummy_input)

print(f"\n입력 텐서 크기: {dummy_input.shape}")
print(f"출력 텐서 크기: {output.shape}")  # 예상 출력: torch.Size([1, 256, 28, 28])

InceptionBlock(
  (pre_conv): Conv2d(3, 192, kernel_size=(1, 1), stride=(1, 1))
  (conv1): Conv2d(192, 64, kernel_size=(1, 1), stride=(1, 1))
  (conv1_2): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1))
  (conv2): Conv2d(96, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv1_3): Conv2d(192, 16, kernel_size=(1, 1), stride=(1, 1))
  (conv3): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (pool): MaxPool2d(kernel_size=3, stride=1, padding=1, dilation=1, ceil_mode=False)
  (conv1_4): Conv2d(192, 32, kernel_size=(1, 1), stride=(1, 1))
)

입력 텐서 크기: torch.Size([1, 3, 28, 28])
출력 텐서 크기: torch.Size([1, 256, 28, 28])
